# 04 — X-IIoTID | Five Model Training & Resource Evaluation

**Objective:** Train and evaluate five ML models separately using the full Core19 dataset split.

**Data:** 574,585 training rows and 123,125 validation rows. **The test split is never opened.**

**Outputs:** Five models, individual validation results, confusion matrices, CPU/RAM/timing results, and short comparison charts.

In [ ]:
from pathlib import Path
from time import perf_counter
from threading import Thread, Event
from datetime import datetime, timezone
import inspect, json, os, gc, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, psutil, sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
    confusion_matrix, precision_score, recall_score, f1_score)

ROOT = Path(r'D:\J26-CS-336-Network-Flow-V2\network_flow_v2')
SPLITS = ROOT / 'data/splits/x_iiotid'
TRAIN_FILE = SPLITS / 'x_iiotid_core19_train_v3.csv.gz'
VALID_FILE = SPLITS / 'x_iiotid_core19_validation_v3.csv.gz'
SPLIT_REPORT = ROOT / 'reports/audits/x_iiotid_v3_split_report.json'
MODEL_DIR = ROOT / 'models/x_iiotid_v3'
AUDITS = ROOT / 'reports/audits'
METRICS = ROOT / 'reports/metrics'
FIGURES = ROOT / 'reports/figures'
SEED = 42
EXPECTED = {'train': 574585, 'validation': 123125, 'test': 123124}
MODEL_NAMES = ['decision_tree','random_forest','extra_trees','lightgbm','adaboost']
CAT = ['Protocol','Service','Conn_state','is_syn_only','Is_SYN_ACK',
       'is_pure_ack','is_with_payload','FIN or RST']
MISSING = {'', '-', '?', 'na', 'n/a', 'nan', 'none', 'null', 'missing', 'unknown', '<missing>'}
SINGLE_FLOW_RUNS = 100   # Only latency sampling; ALL training/validation records are used.

for f in [TRAIN_FILE, VALID_FILE, SPLIT_REPORT]:
    if not f.is_file(): raise FileNotFoundError(f'Missing Notebook 03 output: {f}')
report = json.loads(SPLIT_REPORT.read_text(encoding='utf-8'))
counts = {k:int(v) for k,v in report['counts'].items()}
if counts != EXPECTED: raise ValueError(f'Unexpected frozen split sizes: {counts}')
if report.get('total_rows_preserved', sum(counts.values())) != sum(EXPECTED.values()):
    raise ValueError('Dataset row total does not match the frozen split.')
if report.get('normalized_signature_overlap_groups') != 0:
    raise ValueError('Core19 feature patterns overlap across frozen partitions.')
if report.get('test_scored', False): raise ValueError('Test was reported as scored.')
FEATURES = list(report['features'])
NUM = [f for f in FEATURES if f not in CAT]
if len(FEATURES)!=19 or len(NUM)!=11 or len(CAT)!=8:
    raise ValueError('Unexpected Core19 feature list.')
for d in (MODEL_DIR,AUDITS,METRICS,FIGURES): d.mkdir(parents=True,exist_ok=True)
print(f'Frozen dataset: {sum(counts.values()):,} total rows | Features: {len(FEATURES)}')
print('Training:', counts['train'], '| Validation:', counts['validation'])
print('Test file is NOT opened. Excel files are for manual review only.')

## 1. Load the training and validation datasets

Read all rows. Use `class3` as the label; exclude labels and row IDs from model inputs.

In [ ]:
cols = ['csv_row_index_0based'] + FEATURES + ['class1','class2','class3']
train = pd.read_csv(TRAIN_FILE, dtype=str, keep_default_na=False, low_memory=False)
valid = pd.read_csv(VALID_FILE, dtype=str, keep_default_na=False, low_memory=False)
for part, frame in [('train',train),('validation',valid)]:
    if list(frame.columns) != cols:
        raise ValueError(f'{part}: expected 23 columns in the frozen Core19 order.')
    if len(frame) != EXPECTED[part]:
        raise ValueError(f'{part}: {len(frame)} records; expected {EXPECTED[part]}')
    if frame['csv_row_index_0based'].duplicated().any():
        raise ValueError(f'{part}: duplicate original row identifiers.')
    if not frame['class3'].isin(['Normal','Attack']).all():
        raise ValueError(f'{part}: unexpected class3 label.')
if set(train['csv_row_index_0based']).intersection(valid['csv_row_index_0based']):
    raise ValueError('Train / validation original row IDs overlap.')
y_train = (train['class3'] == 'Attack').astype(np.uint8).to_numpy()
y_valid = (valid['class3'] == 'Attack').astype(np.uint8).to_numpy()
X_train_raw = train[FEATURES].copy()
X_valid_raw = valid[FEATURES].copy()
del train, valid
gc.collect()
print(f'Full training: {len(y_train):,} | Full validation: {len(y_valid):,}')
print('Labels excluded from model features.')

## 2. Prepare features and resource measurement

Convert numerical fields, handle missing values and encode categories. Each model fits its preprocessing on **training data only**.

In [ ]:
def clean_features(frame):
    result = frame[FEATURES].copy()
    for c in NUM:
        values = result[c].astype(str).str.strip()
        values = values.mask(values.str.casefold().isin(MISSING))
        result[c] = pd.to_numeric(values,errors='coerce').replace([np.inf,-np.inf],np.nan).astype(np.float32)
    for c in CAT:
        values = result[c].astype(str).str.strip()
        result[c] = values.mask(values.str.casefold().isin(MISSING),'__MISSING__')
    return result

X_train = clean_features(X_train_raw)
X_valid = clean_features(X_valid_raw)
# Only one cleaned validation row is necessary for timing, not a reduction in evaluation rows.
ONE_FLOW = X_valid.iloc[[0]]

def build_preprocessor():
    enc = {'handle_unknown':'ignore'}
    enc['sparse_output' if 'sparse_output' in inspect.signature(OneHotEncoder).parameters else 'sparse'] = True
    imputer_args = {'strategy':'median'}
    if 'keep_empty_features' in inspect.signature(SimpleImputer).parameters:
        imputer_args['keep_empty_features']=True
    return ColumnTransformer([
        ('numeric',SimpleImputer(**imputer_args),NUM),
        ('category',OneHotEncoder(**enc),CAT)
    ],sparse_threshold=1.0)

def cpu_time_s():
    t = psutil.Process(os.getpid()).cpu_times()
    return t.user + t.system

proc = psutil.Process(os.getpid())
def rss_mib(): return proc.memory_info().rss / (1024**2)

def timed_resource_call(fn):
    initial = rss_mib()
    peak = [initial]
    stop = Event()
    def watch():
        while not stop.wait(0.025):
            peak[0] = max(peak[0], rss_mib())
    thread = Thread(target=watch,daemon=True)
    c0 = cpu_time_s()
    t0 = perf_counter()
    thread.start()
    try:
        result = fn()
    finally:
        elapsed = perf_counter()-t0
        cpu_delta = max(0.0,cpu_time_s()-c0)
        stop.set();thread.join()
        peak[0] = max(peak[0],rss_mib())
    return result, {
        'seconds':round(elapsed,4),
        'process_cpu_seconds':round(cpu_delta,4),
        'avg_process_cpu_pct':round(100*cpu_delta/max(elapsed,1e-9),2),
        'rss_initial_mib':round(initial,2),
        'rss_peak_mib':round(peak[0],2),
        'rss_peak_increase_mib':round(max(0,peak[0]-initial),2)
    }

print('Prepared', X_train.shape, 'train and',X_valid.shape,'validation inputs.')
print('RAM figures include Python, loaded data and libraries; they are not model-only RAM.')

## 3. Shared evaluation function

Every model gets the **same** validation metrics and resource measurements. Each run saves its model and results immediately.

In [ ]:
def evaluate_model(name, estimator):
    print(f'\n===== {name.replace("_"," ").upper()} =====')
    pipeline = Pipeline([
        ('preprocessing',build_preprocessor()),
        ('classifier',estimator)
    ])
    _, train_stats = timed_resource_call(lambda: pipeline.fit(X_train,y_train))
    preds, pred_stats = timed_resource_call(lambda: pipeline.predict(X_valid))
    tn,fp,fn,tp = (int(v) for v in confusion_matrix(y_valid,preds,labels=[0,1]).ravel())
    # Warm up the whole saved preprocessing + model pipeline.
    for _ in range(5): pipeline.predict(ONE_FLOW)
    idx = np.linspace(0,len(X_valid)-1,num=min(SINGLE_FLOW_RUNS,len(X_valid)),dtype=int)
    latency_ms=[]
    for i in idx:
        row = X_valid.iloc[[int(i)]]
        t0=perf_counter()
        pipeline.predict(row)
        latency_ms.append((perf_counter()-t0)*1000)
    model_path = MODEL_DIR / f'{name}_core19_v3.joblib'
    joblib.dump(pipeline,model_path,compress=3)

    metrics = {
        'model':name, 'train_rows':len(y_train), 'validation_rows':len(y_valid),
        'accuracy':round(float(accuracy_score(y_valid,preds)),6),
        'balanced_accuracy':round(float(balanced_accuracy_score(y_valid,preds)),6),
        'precision_attack':round(float(precision_score(y_valid,preds,zero_division=0)),6),
        'recall_attack':round(float(recall_score(y_valid,preds,zero_division=0)),6),
        'f1_attack':round(float(f1_score(y_valid,preds,zero_division=0)),6),
        'false_positive_rate':round(fp/max(tn+fp,1),6),
        'false_negative_rate':round(fn/max(fn+tp,1),6),
        'tn':tn,'fp':fp,'fn':fn,'tp':tp,
    }
    resources = {
        'model':name, 'training_seconds':train_stats['seconds'],
        'train_process_cpu_percent':train_stats['avg_process_cpu_pct'],
        'train_peak_process_ram_mib':train_stats['rss_peak_mib'],
        'train_peak_ram_increase_mib':train_stats['rss_peak_increase_mib'],
        'validation_predict_seconds':pred_stats['seconds'],
        'predict_process_cpu_percent':pred_stats['avg_process_cpu_pct'],
        'predict_peak_process_ram_mib':pred_stats['rss_peak_mib'],
        'batch_throughput_flows_per_second':round(len(y_valid)/max(pred_stats['seconds'],1e-9),1),
        'single_flow_p50_ms':round(float(np.percentile(latency_ms,50)),3),
        'single_flow_p95_ms':round(float(np.percentile(latency_ms,95)),3),
        'single_flow_runs':len(idx),
        'model_size_mib':round(model_path.stat().st_size/(1024**2),4),
    }
    importance = []
    clf = pipeline.named_steps['classifier']
    if hasattr(clf,'feature_importances_'):
        names = pipeline.named_steps['preprocessing'].get_feature_names_out()
        weights = clf.feature_importances_
        importance = [{'feature':str(names[i]),'importance':round(float(weights[i]),6)}
                      for i in np.argsort(weights)[::-1][:10]]
    result = {
        'experiment':'X-IIoTID Core19 V3 — validation only',
        'model_name':name,'created_utc':datetime.now(timezone.utc).isoformat(),
        'model_file':model_path.name,'core19_features':FEATURES,'seed':SEED,
        'metrics':metrics,'resources':resources,
        'top_transformed_feature_importance':importance,
        'python_version':platform.python_version(),
        'sklearn_version':sklearn.__version__,
        'split_report':SPLIT_REPORT.name,
        'test_scored':False,
        'limitations':[
            'Laptop measurements; not measured on Raspberry Pi.',
            'RAM includes the Python process, loaded data and libraries.',
            'CPU percent can exceed 100 for multi-core models.',
            'Single-flow latency includes fitted preprocessing and prediction on already cleaned input.',
            'Capture, feature extraction and full gateway latency are not included.',
            'Dataset signature separation does not prove session/campaign independence.'
        ]
    }
    path = AUDITS / f'x_iiotid_v3_{name}_training.json'
    path.write_text(json.dumps(result,indent=2),encoding='utf-8')

    fig,ax = plt.subplots(figsize=(5.2,4.5))
    image=ax.imshow([[tn,fp],[fn,tp]],cmap='Blues')
    for r in (0,1):
        for c in (0,1):
            ax.text(c,r,f'{[[tn,fp],[fn,tp]][r][c]:,}',ha='center',va='center',
                    color='white' if [[tn,fp],[fn,tp]][r][c] > max(tn,fp,fn,tp)*.55 else 'black')
    ax.set_xticks([0,1],['Normal','Attack'])
    ax.set_yticks([0,1],['Normal','Attack'])
    ax.set_xlabel('Predicted');ax.set_ylabel('Actual')
    ax.set_title(f'{name.replace("_"," ").title()} — validation')
    fig.tight_layout()
    fig.savefig(FIGURES/f'x_iiotid_v3_{name}_confusion_matrix.png',dpi=160,bbox_inches='tight')
    plt.show();plt.close(fig)
    display(pd.DataFrame([metrics]).set_index('model').T)
    display(pd.DataFrame([resources]).set_index('model').T)
    if importance: display(pd.DataFrame(importance).head(10))
    print('Saved:',model_path.name,'|',path.name)
    del pipeline,preds
    gc.collect()
    return result

print('Ready: run each model cell below separately in the listed order.')

## 4. Decision Tree

Fast, small single-tree baseline.

In [ ]:
decision_tree_result = evaluate_model(
    'decision_tree',
    DecisionTreeClassifier(max_depth=14,min_samples_leaf=30,
                           class_weight='balanced',random_state=SEED)
)

## 5. Random Forest

An ensemble of decision trees. Compare accuracy and CPU/RAM cost.

In [ ]:
random_forest_result = evaluate_model(
    'random_forest',
    RandomForestClassifier(n_estimators=100,max_depth=16,min_samples_leaf=20,
                           class_weight='balanced_subsample',random_state=SEED,n_jobs=-1)
)

## 6. Extra Trees

Randomized tree ensemble. Compare missed attacks and false alarms.

In [ ]:
extra_trees_result = evaluate_model(
    'extra_trees',
    ExtraTreesClassifier(n_estimators=100,max_depth=16,min_samples_leaf=20,
                         class_weight='balanced',random_state=SEED,n_jobs=-1)
)

## 7. LightGBM

Gradient-boosted trees. CPU use may exceed 100% because it uses multiple cores.

In [ ]:
from lightgbm import LGBMClassifier
lightgbm_result = evaluate_model(
    'lightgbm',
    LGBMClassifier(n_estimators=160,num_leaves=31,max_depth=-1,
                   learning_rate=0.05,min_child_samples=30,
                   random_state=SEED,n_jobs=4,verbosity=-1)
)

## 8. AdaBoost

Boost many small trees and measure its detection and resource cost.

In [ ]:
small_tree = DecisionTreeClassifier(max_depth=2,min_samples_leaf=30,random_state=SEED)
ada_kwargs = {'n_estimators':100,'learning_rate':0.5,'random_state':SEED}
ada_kwargs['estimator' if 'estimator' in inspect.signature(AdaBoostClassifier).parameters
           else 'base_estimator'] = small_tree
adaboost_result = evaluate_model('adaboost',AdaBoostClassifier(**ada_kwargs))

## 9. Compare all five models

Read each saved model result. This step can be rerun without retraining the models.

In [ ]:
saved = []
for name in MODEL_NAMES:
    file = AUDITS / f'x_iiotid_v3_{name}_training.json'
    if not file.is_file(): raise FileNotFoundError(f'Run the {name} cell first: {file}')
    data = json.loads(file.read_text(encoding='utf-8'))
    if data['metrics']['train_rows'] != EXPECTED['train'] or data['metrics']['validation_rows'] != EXPECTED['validation']:
        raise ValueError(f'Unexpected saved data sizes for {name}.')
    if data.get('test_scored') is not False: raise ValueError(f'Test scoring claim inconsistent for {name}.')
    saved.append(data)
comparison = pd.DataFrame([item['metrics'] for item in saved])
resources = pd.DataFrame([item['resources'] for item in saved])
comparison.to_csv(METRICS/'x_iiotid_v3_five_model_validation.csv',index=False)
resources.to_csv(METRICS/'x_iiotid_v3_five_model_resources.csv',index=False)
summary = {
    'experiment':'X-IIoTID Core19 V3 five-model comparison',
    'training_rows':EXPECTED['train'],'validation_rows':EXPECTED['validation'],
    'test_rows_reserved':EXPECTED['test'],'test_scored':False,
    'individual_json':[f'x_iiotid_v3_{n}_training.json' for n in MODEL_NAMES],
    'note':'Laptop resource measurements include other process memory; end-to-end gateway metrics still pending.'
}
(AUDITS/'x_iiotid_v3_model_training.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')

plot = comparison.copy()
plot['name'] = plot['model'].str.replace('_',' ').str.title()
fig,axes = plt.subplots(1,2,figsize=(12,4.5))
axes[0].bar(plot['name'],100*plot['f1_attack'])
axes[0].set_title('Attack F1-score (validation)');axes[0].set_ylabel('%')
axes[0].set_ylim(0,105)
axes[1].bar(plot['name'],100*plot['false_positive_rate'])
axes[1].set_title('False-positive rate (validation)');axes[1].set_ylabel('%')
for ax in axes:
    ax.tick_params(axis='x',labelrotation=20)
    ax.grid(axis='y',alpha=0.2)
fig.tight_layout(); fig.savefig(FIGURES/'x_iiotid_v3_model_f1.png',dpi=170,bbox_inches='tight')
plt.show();plt.close(fig)

r=resources.copy();r['name']=r['model'].str.replace('_',' ').str.title()
fig,axes=plt.subplots(2,2,figsize=(12,8))
for ax,(column,title,unit) in zip(axes.flat,[
    ('training_seconds','Training time','seconds'),
    ('train_peak_process_ram_mib','Peak process RAM during training','MiB'),
    ('single_flow_p95_ms','Single-flow inference p95','ms'),
    ('model_size_mib','Saved model size','MiB')]):
    ax.bar(r['name'],r[column]);ax.set_title(title);ax.set_ylabel(unit)
    ax.tick_params(axis='x',labelrotation=15);ax.grid(axis='y',alpha=0.2)
fig.tight_layout();fig.savefig(FIGURES/'x_iiotid_v3_model_resources.png',dpi=170,bbox_inches='tight')
plt.show();plt.close(fig)

display(comparison[['model','accuracy','precision_attack','recall_attack','f1_attack',
                    'false_positive_rate','false_negative_rate','tn','fp','fn','tp']])
display(resources[['model','training_seconds','train_process_cpu_percent',
                   'train_peak_process_ram_mib','validation_predict_seconds',
                   'batch_throughput_flows_per_second','single_flow_p95_ms','model_size_mib']])
print('Saved: 5 individual JSON reports, 5 confusion matrices, 2 comparison CSVs and 2 charts.')
print('Test split was never read.')